# 03 - Prompt Engineering

Generate 40 synthetic AG News briefs with basic and advanced prompts using local Qwen3-4B. Compare format, length, and lexical diversity.

In [1]:
from pathlib import Path
import hashlib
import itertools
import json
import re
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "prompts" / "basic.txt").exists():
            return candidate
    raise FileNotFoundError("Run Jupyter from the repository or its notebooks directory.")


ROOT = find_project_root()
PROMPTS_DIR = ROOT / "prompts"
DATA_OUT = ROOT / "data" / "synthetic" / "task006"
RESULTS_OUT = ROOT / "results" / "task006"
DATA_OUT.mkdir(parents=True, exist_ok=True)
RESULTS_OUT.mkdir(parents=True, exist_ok=True)

MODEL_ID = "qwen3:4b-instruct"
API_URL = "http://127.0.0.1:11434"
SEED = 42
N_PER_CATEGORY = 5
MAX_NEW_TOKENS = 180

CATEGORIES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
SCENARIOS = {
    "World": [
        "regional diplomatic talks in East Africa",
        "election administration in Southeast Asia",
        "cross-border transport agreement in Europe",
        "flood relief coordination in South America",
        "trade negotiations in the Middle East",
    ],
    "Sports": [
        "regional football final", "tennis qualifying upset",
        "athletics championship selection", "basketball coaching change",
        "cricket tournament opener",
    ],
    "Business": [
        "small manufacturer earnings", "retail chain expansion",
        "labor contract negotiations", "agricultural commodity prices",
        "competition review of a proposed merger",
    ],
    "Sci/Tech": [
        "medical device prototype", "software security update",
        "small satellite mission", "battery materials research",
        "AI tool deployment at a university",
    ],
}
assert all(len(SCENARIOS[category]) == N_PER_CATEGORY for category in CATEGORIES.values())

prompts = {
    "system": "You write original, plausible, fictional English news briefs for a text-classification research dataset. Do not reproduce or claim to report a verified real event. Return only two lines headed exactly 'Headline:' and 'Summary:'. Do not add explanations, markdown, or category labels.",
    "basic": (PROMPTS_DIR / "basic.txt").read_text(encoding="utf-8").strip(),
    "advanced": (PROMPTS_DIR / "advanced.txt").read_text(encoding="utf-8").strip(),
}
prompt_sha256 = hashlib.sha256(
    json.dumps(prompts, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
).hexdigest()
display(pd.DataFrame({"strategy": ["basic", "advanced"],
                      "prompt": [prompts["basic"], prompts["advanced"]]}))

,strategy,prompt
0,basic,Write one short AG News-style news item in the...
1,advanced,Write one original AG News-style news item in ...


In [2]:
plan = pd.DataFrame([
    {
        "sample_id": f"{strategy}_{label}_{replicate + 1}",
        "strategy": strategy,
        "label": label,
        "category": category,
        "replicate": replicate + 1,
        "seed": SEED + label * 100 + replicate,
        "scenario": SCENARIOS[category][replicate] if strategy == "advanced" else None,
    }
    for strategy, (label, category), replicate in itertools.product(
        ("basic", "advanced"), CATEGORIES.items(), range(N_PER_CATEGORY)
    )
])
assert len(plan) == 40 and not plan.sample_id.duplicated().any()
display(plan.groupby(["strategy", "category"]).size().unstack())

category,Business,Sci/Tech,Sports,World
strategy,,,,
advanced,5,5,5,5
basic,5,5,5,5


In [3]:
import requests

session = requests.Session()
tags_response = session.get(f"{API_URL}/api/tags", timeout=10)
tags_response.raise_for_status()
model_record = next(
    (item for item in tags_response.json()["models"] if item["name"] == MODEL_ID), None
)
if model_record is None:
    raise RuntimeError(f"Install the local model first: ollama pull {MODEL_ID}")
MODEL_DIGEST = model_record["digest"]
OLLAMA_VERSION = session.get(f"{API_URL}/api/version", timeout=10).json().get("version")
print({"model": MODEL_ID, "digest": MODEL_DIGEST,
       "quantization": model_record.get("details", {}).get("quantization_level"),
       "ollama_version": OLLAMA_VERSION})

{'model': 'qwen3:4b-instruct', 'digest': '0edcdef34593eac1aa2be9c7d06c432dcf81945adca5eca2f27662c18f168ba0', 'quantization': 'Q4_K_M', 'ollama_version': '0.40.11'}


In [4]:
manifest_path = RESULTS_OUT / "manifest.json"
samples_path = DATA_OUT / "samples.jsonl"
manifest = {
    "backend": "Ollama",
    "model_id": MODEL_ID,
    "model_digest": MODEL_DIGEST,
    "model_variant": "Qwen3-4B-Instruct-2507 Q4_K_M",
    "ollama_version": OLLAMA_VERSION,
    "prompt_sha256": prompt_sha256,
    "prompt_files": ["prompts/basic.txt", "prompts/advanced.txt"],
    "seed": SEED,
    "n_per_category": N_PER_CATEGORY,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "think": False,
}
if manifest_path.exists():
    previous = json.loads(manifest_path.read_text(encoding="utf-8"))
    if previous != manifest:
        raise RuntimeError("Existing TASK-006 output uses different settings. Move it before a new run.")
else:
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")


def parse_article(raw: str) -> tuple[str, str, bool]:
    match = re.fullmatch(r"\s*Headline:\s*([^\n]+)\s*\nSummary:\s*(.+?)\s*",
                         raw, flags=re.IGNORECASE | re.DOTALL)
    if match is None:
        return "", "", False
    return match.group(1).strip(), match.group(2).strip(), True


def generate_one(item: dict) -> dict:
    content = prompts[item["strategy"]].format(
        category=item["category"], scenario=item["scenario"]
    )
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": prompts["system"]},
            {"role": "user", "content": content},
        ],
        "stream": False,
        "think": False,
        "options": {
            "seed": item["seed"], "num_predict": MAX_NEW_TOKENS,
            "temperature": 0.7, "top_p": 0.8, "top_k": 20,
        },
    }
    start = time.perf_counter()
    response = session.post(f"{API_URL}/api/chat", json=payload, timeout=300)
    response.raise_for_status()
    result = response.json()
    elapsed = time.perf_counter() - start
    raw = result["message"]["content"].strip()
    headline, summary, valid = parse_article(raw)
    return {
        **item, "prompt": content, "raw_output": raw,
        "headline": headline, "summary": summary,
        "text": f"{headline} {summary}".strip() if valid else raw,
        "format_valid": valid, "new_tokens": result.get("eval_count"),
        "seconds": round(elapsed, 3), "done_reason": result.get("done_reason"),
    }


existing = []
if samples_path.exists():
    existing = [json.loads(line) for line in samples_path.read_text(encoding="utf-8").splitlines() if line]
done = {row["sample_id"] for row in existing}
if len(done) != len(existing) or not done.issubset(set(plan.sample_id)):
    raise RuntimeError("Existing samples contain duplicate or unexpected IDs.")

for item in plan.to_dict("records"):
    if item["sample_id"] in done:
        continue
    result = generate_one(item)
    with samples_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")
    done.add(item["sample_id"])
    print(f"{len(done):02d}/40 {item['sample_id']} ? {result['seconds']:.1f}s; format={result['format_valid']}")

samples = pd.read_json(samples_path, lines=True)
assert len(samples) == 40 and not samples.sample_id.duplicated().any()
display(samples[["sample_id", "category", "headline", "summary", "format_valid"]].head())

,sample_id,category,headline,summary,format_valid
0,basic_0_1,World,Global Climate Summit Ends with Historic Agree...,Leaders from 195 countries reached a landmark ...,True
1,basic_0_2,World,UN Calls for Global Action on Climate-Induced ...,The United Nations has urged nations to streng...,True
2,basic_0_3,World,UN Warns of Climate Threat to Global Food Secu...,The United Nations has issued a stark warning ...,True
3,basic_0_4,World,Global Leaders Convene in Geneva to Address Cl...,Diplomats from 120 countries gathered in Genev...,True
4,basic_0_5,World,UN Appeals for Global Aid as Sahel Drought Wor...,A severe drought in the Sahel region has displ...,True


Samples are saved to `data/synthetic/task006/samples.jsonl`. Rerunning this cell skips completed samples.

In [5]:
WORD = re.compile(r"[^\W\d_]+(?:'[^\W\d_]+)?", flags=re.UNICODE)


def words(text: str) -> list[str]:
    return WORD.findall(text.lower())


def distinct_2(texts: list[str]) -> float:
    bigrams = [pair for text in texts for pair in zip(words(text), words(text)[1:])]
    return len(set(bigrams)) / len(bigrams) if bigrams else float("nan")


def mean_pairwise_similarity(texts: list[str]) -> float:
    if len(texts) < 2 or not any(text.strip() for text in texts):
        return float("nan")
    try:
        matrix = TfidfVectorizer(stop_words="english").fit_transform(texts)
    except ValueError:  # all texts may contain only stop words or punctuation
        return float("nan")
    sims = cosine_similarity(matrix)
    return float(sims[np.triu_indices(len(texts), k=1)].mean())


samples["word_count"] = samples.text.map(lambda text: len(words(text)))
samples["length_valid"] = samples.word_count.between(35, 80)
samples["exact_duplicate"] = samples.duplicated(["strategy", "category", "text"], keep=False)
samples.to_csv(RESULTS_OUT / "sample_metrics.csv", index=False, encoding="utf-8")

comparison = (
    samples.groupby(["strategy", "category"], sort=False)
    .agg(n=("sample_id", "size"), format_rate=("format_valid", "mean"),
         length_rate=("length_valid", "mean"), mean_words=("word_count", "mean"),
         duplicate_rate=("exact_duplicate", "mean"), mean_seconds=("seconds", "mean"))
    .reset_index()
)
diversity = samples.groupby(["strategy", "category"], sort=False).text.apply(list).reset_index()
diversity["distinct_2"] = diversity.text.map(distinct_2)
diversity["mean_tfidf_cosine"] = diversity.text.map(mean_pairwise_similarity)
first_35 = diversity.text.map(lambda texts: [" ".join(words(text)[:35]) for text in texts])
diversity["distinct_2_first35"] = first_35.map(distinct_2)
diversity["mean_tfidf_cosine_first35"] = first_35.map(mean_pairwise_similarity)
comparison = comparison.merge(diversity.drop(columns="text"), on=["strategy", "category"])
comparison.to_csv(RESULTS_OUT / "comparison.csv", index=False, encoding="utf-8")
display(comparison.round(3))

,strategy,category,n,format_rate,length_rate,mean_words,duplicate_rate,mean_seconds,distinct_2,mean_tfidf_cosine,distinct_2_first35,mean_tfidf_cosine_first35
0,basic,World,5,1.0,1.0,49.2,0.0,1.943,0.917,0.103,0.935,0.092
1,basic,Sports,5,1.0,1.0,49.2,0.0,1.340,0.855,0.098,0.835,0.111
2,basic,Business,5,1.0,1.0,49.4,0.0,1.228,0.769,0.220,0.706,0.261
3,basic,Sci/Tech,5,1.0,1.0,54.4,0.0,1.216,0.768,0.193,0.688,0.228
4,advanced,World,5,1.0,1.0,63.6,0.0,1.495,0.946,0.037,0.941,0.017
5,advanced,Sports,5,1.0,0.8,70.8,0.0,2.023,0.917,0.019,0.912,0.017
6,advanced,Business,5,1.0,1.0,62.0,0.0,1.544,0.951,0.033,0.941,0.011
7,advanced,Sci/Tech,5,1.0,1.0,65.0,0.0,1.605,0.941,0.040,0.924,0.029


Metrics are descriptive for five samples per category.

In [6]:
review_path = RESULTS_OUT / "review_scored.csv"
review = pd.read_csv(review_path)
assert len(review) == len(samples) == 40
assert not review.sample_id.duplicated().any()
assert set(review.sample_id) == set(samples.sample_id)

checked = samples[["sample_id", "strategy", "category", "headline", "summary"]].merge(
    review, on="sample_id", suffixes=("_generated", "_review"), validate="one_to_one"
)
for field in ("strategy", "category", "headline", "summary"):
    assert (checked[f"{field}_generated"] == checked[f"{field}_review"]).all(), field
for field in ("category_fit_1to5", "realism_1to5"):
    review[field] = pd.to_numeric(review[field], errors="raise")
    assert review[field].notna().all() and review[field].between(1, 5).all()

quality = review.groupby(["strategy", "category"], as_index=False).agg(
    reviewed_n=("sample_id", "size"),
    category_fit_1to5=("category_fit_1to5", "mean"),
    realism_1to5=("realism_1to5", "mean"),
)
final_comparison = comparison.merge(
    quality, on=["strategy", "category"], validate="one_to_one"
)
columns = ["strategy", "category", "n", "format_rate", "length_rate",
           "distinct_2_first35", "mean_tfidf_cosine_first35",
           "category_fit_1to5", "realism_1to5"]
display(final_comparison[columns].round(3))

overall = final_comparison.groupby("strategy")[[
    "format_rate", "length_rate", "distinct_2_first35",
    "mean_tfidf_cosine_first35", "category_fit_1to5", "realism_1to5"
]].mean().round(3)
display(overall)

,strategy,category,n,format_rate,length_rate,distinct_2_first35,mean_tfidf_cosine_first35,category_fit_1to5,realism_1to5
0,basic,World,5,1.0,1.0,0.935,0.092,5.0,3.4
1,basic,Sports,5,1.0,1.0,0.835,0.111,5.0,2.2
2,basic,Business,5,1.0,1.0,0.706,0.261,4.8,3.2
3,basic,Sci/Tech,5,1.0,1.0,0.688,0.228,5.0,3.2
4,advanced,World,5,1.0,1.0,0.941,0.017,5.0,2.2
5,advanced,Sports,5,1.0,0.8,0.912,0.017,5.0,2.2
6,advanced,Business,5,1.0,1.0,0.941,0.011,5.0,3.6
7,advanced,Sci/Tech,5,1.0,1.0,0.924,0.029,5.0,3.6


,format_rate,length_rate,distinct_2_first35,mean_tfidf_cosine_first35,category_fit_1to5,realism_1to5
strategy,,,,,,
advanced,1.0,0.95,0.929,0.018,5.00,2.9
basic,1.0,1.00,0.791,0.173,4.95,3.0


All 40 articles were reviewed. The advanced prompt increases lexical diversity (distinct-2: 0.929 vs 0.791) but not overall realism (2.9 vs 3.0/5); World realism falls from 3.4 to 2.2/5.